# NUERONCE foundational-recovery Colab journal

Trains the **`base_35m`** rung (34.4M params, fresh init -- shapes don't match
`chat_11m`, so there is no weight transfer) on a corpus-stack pull, then runs
**ForgeLoop SFT** on the already-cleaned `data/foundational_recovery_v3_1`
rows with the byte-identical system prompt the sealed gate uses, then scores
the result against this repo's own acceptance bar
(`docs/CODEX_HANDOFF.md`, `FOUNDATIONAL_GENERATION_RECOVERY.md`):

1. `base_35m` held-out bpb <= 1.5
2. choice-ranking accuracy beats chance by >=15 pts on >=3 MCQ subjects
3. phase2 inference suite >= 60% valid / non-echo / stop-terminated
4. 5/5 raw transcripts are grammatical English addressing the question
5. the **sealed 8-item proof gate**, run once, unmodified

**Ground rules carried over from the recovery doc -- do not violate these:**
- Never edit `scripts/eval_foundational_proof_gate.py`'s 8 sealed cases to
  make the gate pass. Run it once per checkpoint and record the number.
- The corpus mix below caps every single source near 25% of bytes (the
  77%-arithmetic-poisoning lesson in `docs/RESULTS.md`) -- code is one
  register among several, not the bulk of the corpus.
- `base_35m` is a **fresh-init** run. There is no upcycling path from
  `chat_11m` in this repo; `train_checkpoint.py` will refuse to `--resume`
  across a config mismatch.
- Everything here is resumable across Colab disconnects: checkpoints are
  written atomically, backed up to Drive every 30 minutes, and every
  training subprocess can be relaunched with `--resume`.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine for
`base_35m`). This notebook targets the `nueronce` package (the repo formerly
named `cfna` was renamed -- do not point this at old `cfna_*` branches or
checkpoints, they are not resume-compatible).


In [ ]:
# 1) Clone/reset to the live repository branch and install dependencies.
from pathlib import Path
import os, subprocess, sys

REPO = "LMMinier/nueronce"
BRANCH = "fix/foundational-generation-recovery"  # has the sealed-gate + recovery scripts merged in
REPO_DIR = Path("/content/nueronce") if Path("/content").exists() else Path.cwd()

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    f"https://github.com/{REPO}.git", str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "datasets", "tqdm", "pytest"], check=True)

HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import torch, nueronce
print("branch:", BRANCH)
print("commit:", HEAD)
print("nueronce:", nueronce.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0),
          "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("Enable a GPU runtime before training (Runtime -> Change runtime type).")


In [ ]:
# 2) Safety gates before any fp16/AMP training + confirm the sealed-gate
#    pipeline tests (byte-prefix equality, no state leak, etc.) are green
#    on this exact commit before spending GPU hours on it.
subprocess.run([sys.executable, "-m", "pytest",
                "tests/test_gpu_amp.py",
                "tests/test_foundational_recovery_pipeline.py",
                "-q"], check=True)


In [ ]:
# 3) Mount Drive. Checkpoints/corpus are keyed to the commit + config, so a
#    stale cache from an older architecture can't silently get reused.
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
CKPT_ROOT = DRIVE / "nueronce_checkpoints"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = f"base35m_{HEAD[:12]}"
CORPUS_DIR = Path(f"/content/corpus_large_{VERSION}")
DRIVE_CORPUS = DRIVE / f"corpus_large_{VERSION}"
BASE_CKPT = CKPT_ROOT / f"nueronce_base_35m_{VERSION}.pt"
SFT_CKPT = CKPT_ROOT / f"nueronce_forgeloop_sft_{VERSION}.pt"
SFT_BEST = SFT_CKPT.with_name(SFT_CKPT.stem + "_best.pt")

BASE_LOG = Path("/content/base_train.log")
SFT_LOG = Path("/content/sft_train.log")

SYSTEM_PROMPT_FILE = REPO_DIR / "runs/forgeloop/system_prompt.txt"
SFT_DATA_DIR = REPO_DIR / "data/foundational_recovery_v3_1"  # already cleaned + committed

print("version:", VERSION)
print("corpus dir:", CORPUS_DIR)
print("base checkpoint:", BASE_CKPT)
print("sft checkpoint:", SFT_CKPT)
print("system prompt:", SYSTEM_PROMPT_FILE.read_text(encoding="utf-8"))


## 4) Build the rebalanced corpus

`the_stack_smol` (code) is capped at 20% of bytes here, not left to dominate
the mix like the old night-session run (452MB/907MB, ~50%) -- the repo's own
25%-per-source rule (`docs/LOCAL_TRAINING_PLAYBOOK.md`) exists precisely
because one source at 77% already poisoned an earlier SFT pass. Adjust
SOURCES below to change the mix; keep every value under ~25% of the total.


In [ ]:
# 4) Rebalanced multi-subject corpus. Each source is pulled into its own temp
#    directory (so dump_corpus_stack.py's per-run manifest.jsonl don't clobber
#    each other), then merged into one corpus_large/ manifest.
import json, shutil

REBUILD_CORPUS = False  # True forces a rebuild even if Drive has this version cached

SOURCES = {                              # source_id -> target bytes
    "cosmopedia_100k":          100_000_000,  # 20% -- synthetic educational prose
    "english_wikipedia_latest": 100_000_000,  # 20% -- general knowledge
    "open_web_math":             75_000_000,  # 15% -- math
    "the_stack_smol":           100_000_000,  # 20% -- ~30-language permissive code
    "project_gutenberg":         75_000_000,  # 15% -- literature
    "pmc_oa_comm":               50_000_000,  # 10% -- health/medicine
}                                             # total 500 MB, no source over 20%

if not REBUILD_CORPUS and DRIVE_CORPUS.exists() and (DRIVE_CORPUS / "manifest.jsonl").exists():
    if CORPUS_DIR.exists():
        shutil.rmtree(CORPUS_DIR)
    shutil.copytree(DRIVE_CORPUS, CORPUS_DIR)
    print("restored corpus from Drive:", CORPUS_DIR)
else:
    if CORPUS_DIR.exists():
        shutil.rmtree(CORPUS_DIR)
    (CORPUS_DIR / "stack_text").mkdir(parents=True, exist_ok=True)
    merged_records = []
    for source_id, target_bytes in SOURCES.items():
        tmp_dir = Path(f"/content/_corpus_tmp_{source_id}")
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/dump_corpus_stack.py"),
               "--out", str(tmp_dir), "--sources", source_id,
               "--target-bytes", str(target_bytes), "--val-every", "20"]
        print(" ".join(cmd))
        subprocess.run(cmd, cwd=REPO_DIR, check=True)
        for f in (tmp_dir / "stack_text").glob("*"):
            shutil.copy2(f, CORPUS_DIR / "stack_text" / f.name)
        manifest_path = tmp_dir / "manifest.jsonl"
        if manifest_path.exists():
            merged_records += [json.loads(l) for l in manifest_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        shutil.rmtree(tmp_dir)

    with (CORPUS_DIR / "manifest.jsonl").open("w", encoding="utf-8") as fh:
        for rec in merged_records:
            fh.write(json.dumps(rec) + "\n")

    if DRIVE_CORPUS.exists():
        shutil.rmtree(DRIVE_CORPUS)
    shutil.copytree(CORPUS_DIR, DRIVE_CORPUS)
    print("built + saved corpus to Drive:", DRIVE_CORPUS)


In [ ]:
# 5) Corpus audit: per-source share, split sizes, and the 25%-cap check.
from collections import Counter, defaultdict

records = [json.loads(l) for l in (CORPUS_DIR / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
assert records, "corpus manifest is empty"

splits = Counter(r["split"] for r in records)
source_bytes = defaultdict(int)
for r in records:
    source_bytes[r["source_collection"]] += int(r.get("n_bytes", 0))
total_bytes = sum(source_bytes.values())

print("records:", len(records), "| splits:", dict(splits), "| total:", f"{total_bytes/1e6:.1f} MB")
print()
for name, n in sorted(source_bytes.items(), key=lambda kv: -kv[1]):
    share = n / total_bytes
    flag = "  <-- ABOVE 25% CAP" if share > 0.25 else ""
    print(f"{name:30s} {n/1e6:7.1f} MB  {share*100:5.1f}%{flag}")

assert splits["val"] >= 2, "need at least 2 held-out documents for a real bpb signal"
assert all(source_bytes[n] / total_bytes <= 0.30 for n in source_bytes), "a source exceeded the poisoning-risk cap"


## 6) Base pretraining -- `base_35m`, fresh init

Runs as a background process. 6c is the main loop -- it polls, prints
progress, backs up to Drive, and stops cleanly (SIGINT) once
`BASE_TRAIN_MINUTES` is spent, so this is safe to leave in a "Run All"
-- it is bounded and does not need manual babysitting. 6a/6b still work as
optional on-demand peeks (rerun anytime) if you're driving the notebook by
hand instead; 6b is worth checking manually for the repetition-collapse
failure signature already documented on this architecture. Note
`train_checkpoint.py` keeps only the *latest* snapshot -- there is no
separate best.pt for base pretraining (unlike the SFT trainer below), so a
regression after the true minimum bpb is not recoverable mid-run.


In [ ]:
# 6) Launch base pretraining as a background process.
import time

BASE_TRAIN_MINUTES = 60 * 8  # adjust per session; rerun this cell (same BASE_CKPT) to continue

def checkpoint_is_valid(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        return "state_dict" in ck and "config" in ck
    except Exception as exc:
        print("checkpoint failed to load, will fresh-init instead:", repr(exc))
        return False

base_cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/train_checkpoint.py"),
            "--preset", "base_35m", "--corpus", str(CORPUS_DIR),
            "--minutes", str(BASE_TRAIN_MINUTES), "--seq", "192", "--batch", "16",
            "--lr", "3e-4", "--amp", "--device", "cuda", "--out", str(BASE_CKPT)]
if checkpoint_is_valid(BASE_CKPT):
    base_cmd.append("--resume")

log_fh = open(BASE_LOG, "a")
BASE_PROC = subprocess.Popen(base_cmd, cwd=REPO_DIR, stdout=log_fh, stderr=subprocess.STDOUT)
print("launched base training, PID", BASE_PROC.pid)
print("log:", BASE_LOG, "| checkpoint:", BASE_CKPT)


In [ ]:
# 6a) Monitor the background run (rerun any time).
import re
alive = BASE_PROC.poll() is None
tail = BASE_LOG.read_text(errors="replace").splitlines()[-15:] if BASE_LOG.exists() else []
print("training alive:", alive, "" if alive else f"(exit code {BASE_PROC.returncode})")
print("\n".join(tail))
bpbs = [float(m.group(1)) for l in BASE_LOG.read_text(errors="replace").splitlines()
        if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
if bpbs:
    print(f"\nheld-out bpb: start {bpbs[0]:.3f} -> now {bpbs[-1]:.3f} (best {min(bpbs):.3f}; target <= 1.5)")


In [ ]:
# 6b) Live inference probe -- loads the checkpoint on CPU so it never
#     touches the GPU mid-step; atomic writes make reads always consistent.
#     Watch specifically for the documented failure signature: repetition
#     loops ("the state of the state of the...") or collapse to whitespace.
from nueronce.chat import load_checkpoint

if not BASE_CKPT.exists():
    print("no checkpoint yet")
else:
    model, _ = load_checkpoint(str(BASE_CKPT))
    model = model.to("cpu").eval()

    @torch.no_grad()
    def complete_bytes(prompt: str, max_new: int = 100, max_ctx: int = 256) -> str:
        ids = list(prompt.encode("utf-8"))
        out = bytearray()
        for _ in range(max_new):
            ctx = torch.tensor([ids[-max_ctx:]], dtype=torch.long)
            logits, _ = model(ctx)
            nxt = int(logits[0, -1].argmax())
            ids.append(nxt)
            out.append(nxt)
        return out.decode("utf-8", errors="replace")

    for p in ["The nature of human understanding is", "Once upon a time", "def add(a, b):\n"]:
        print(f">>> {p!r}")
        print(complete_bytes(p))
        print()


In [ ]:
# 6c) Poll until base training finishes or BASE_TRAIN_MINUTES is spent --
#     Run-All safe: prints progress and backs up to Drive automatically, no
#     manual re-running of 6a/6b required. Sends SIGINT (not SIGKILL) on
#     timeout so the trainer exits through its normal save path; falls back
#     to terminate() only if it doesn't stop within the grace period.
import re, signal

POLL_SECONDS = 600
BACKUP_SECONDS = 1800
STOP_GRACE_SECONDS = 60

start = time.time()
last_backup = start
while True:
    try:
        rc = BASE_PROC.wait(timeout=POLL_SECONDS)
        print(f"base training finished on its own, exit code {rc}")
        break
    except subprocess.TimeoutExpired:
        pass

    elapsed_min = (time.time() - start) / 60
    log_text = BASE_LOG.read_text(errors="replace") if BASE_LOG.exists() else ""
    bpbs = [float(m.group(1)) for l in log_text.splitlines() if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
    trend = f"best {min(bpbs):.3f}, last {bpbs[-1]:.3f}" if bpbs else "no evals logged yet"
    print(f"[{elapsed_min:6.1f} min] {trend}")

    if time.time() - last_backup >= BACKUP_SECONDS and BASE_CKPT.exists():
        shutil.copy2(BASE_CKPT, CKPT_ROOT / BASE_CKPT.name)
        last_backup = time.time()
        print("    auto-backup ->", CKPT_ROOT / BASE_CKPT.name)

    if elapsed_min >= BASE_TRAIN_MINUTES:
        print(f"time budget ({BASE_TRAIN_MINUTES} min) reached -- stopping training cleanly")
        BASE_PROC.send_signal(signal.SIGINT)
        try:
            rc = BASE_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"stopped cleanly, exit code {rc}")
        except subprocess.TimeoutExpired:
            BASE_PROC.terminate()
            rc = BASE_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"forced stop after grace period, exit code {rc}")
        break

shutil.copy2(BASE_CKPT, CKPT_ROOT / BASE_CKPT.name)
ck = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)
history = ck.get("history", [])
best_bpb = min((h["heldout_bpb"] for h in history), default=float("nan"))
print(f"\nstep {ck.get('step')} | best held-out bpb {best_bpb:.4f} | target <= 1.5: "
      f"{'PASS' if best_bpb <= 1.5 else 'not yet -- rerun cell 6 to keep training'}")


## 6.5) Scale up the SFT set

`data/foundational_recovery_v3_1` alone is 3,684 rows total (2,574 train) --
`docs/RESULTS.md` names this exact dataset by row count as too small to
generalize to unseen phrasing on the sealed gate: *"3,684 SFT rows on ~14 MB
of pretraining cannot provide"* that. Per `docs/CODEX_HANDOFF.md`'s fix, mix
in `nueronce.training.synthetic_dialogue` (~127K template-generated pairs)
and `nueronce.training.mcq_sft` (ARC/OpenBookQA/CommonsenseQA/MathQA/GSM8K),
capped per category/subject -- raw `synthetic_dialogue` output is 77.6%
arithmetic before capping, the exact poisoning shape `docs/RESULTS.md`
already warns about, measured below rather than assumed away. The original
v3.1 `val.jsonl`/`test.jsonl` stay untouched as the held-out set -- only
`train.jsonl` grows, and the SFT launch cell below samples domains uniformly
(`--balanced-domain-sampling`) rather than relying on row-count ratios.


In [ ]:
# 6.5) Build the scaled-up SFT training file: the v3.1 ForgeLoop rows kept
#      whole, plus capped synthetic_dialogue and capped MCQ subjects. val/test
#      are left as the original v3.1 files -- nothing new leaks into
#      validation, and gate_ready / response_argmax_acc stay comparable to
#      the small-data run.
import hashlib
import random
from collections import Counter

from nueronce.training.synthetic_dialogue import generate_all
from nueronce.training.mcq_sft import load_and_convert
from nueronce.corpus.stack import get_entry

SYNTHETIC_CATEGORY_CAP = 8000
MCQ_SUBJECT_CAP = 8000
MCQ_SUBJECTS_FOR_SFT = ["arc_easy", "arc_challenge", "openbookqa", "commonsense_qa", "math_qa", "gsm8k"]
SCALED_TRAIN = SFT_DATA_DIR.parent / "foundational_recovery_v3_1_scaled" / "train.jsonl"
SCALED_TRAIN.parent.mkdir(parents=True, exist_ok=True)


def _record(prompt, response, domain):
    return {"prompt": prompt, "response": response, "domain": domain}


seen = set()
rows = []

# a) the existing cleaned ForgeLoop curriculum, kept whole -- this is the
#    software-engineering/ForgeLoop-specific coverage the sealed gate itself
#    draws its 8 cases from the same distribution as.
for line in (SFT_DATA_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    row = dict(json.loads(line))
    row["domain"] = "forgeloop"
    key = hashlib.sha256((row["prompt"] + "|" + row["response"]).encode("utf-8")).hexdigest()
    if key not in seen:
        seen.add(key)
        rows.append(row)
print(f"forgeloop v3.1: {len(rows)} rows")

# b) synthetic_dialogue, capped per category.
by_category = {}
for rec in generate_all():
    by_category.setdefault(rec["category"], []).append(rec)
for category, recs in sorted(by_category.items()):
    added = 0
    for rec in recs[:SYNTHETIC_CATEGORY_CAP]:
        prompt, response = rec["messages"][0]["content"], rec["messages"][1]["content"]
        key = hashlib.sha256((prompt + "|" + response).encode("utf-8")).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_record(prompt, response, f"synthetic_{category}"))
        added += 1
    print(f"synthetic_dialogue/{category:22s} {added:6d} / {len(recs):6d} available (cap {SYNTHETIC_CATEGORY_CAP})")

# c) MCQ choice-ranking + free-form QA, capped per subject.
for subject in MCQ_SUBJECTS_FOR_SFT:
    recs = list(load_and_convert(get_entry(subject), split="train", limit=MCQ_SUBJECT_CAP))
    added = 0
    for rec in recs:
        prompt, response = rec["messages"][0]["content"], rec["messages"][1]["content"]
        key = hashlib.sha256((prompt + "|" + response).encode("utf-8")).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_record(prompt, response, f"mcq_{subject}"))
        added += 1
    print(f"mcq_sft/{subject:18s} {added:6d} rows (cap {MCQ_SUBJECT_CAP})")

random.Random(1234).shuffle(rows)
with SCALED_TRAIN.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row) + "\n")

domain_counts = Counter(row["domain"] for row in rows)
print(f"\ntotal unique rows: {len(rows)}  ->  {SCALED_TRAIN}")
for domain, n in domain_counts.most_common():
    share = n / len(rows)
    flag = "  <-- ABOVE 25% CAP" if share > 0.25 else ""
    print(f"  {domain:24s} {n:6d}  {share*100:5.1f}%{flag}")


## 7) ForgeLoop SFT on the cleaned v3.1 rows

Uses the scaled `train.jsonl` built in 6.5 (v3.1 ForgeLoop rows + capped
synthetic_dialogue + capped MCQ subjects), the original v3.1 `val.jsonl` as
held-out validation, and the
**exact same system-prompt file** the sealed gate evaluates with
(`runs/forgeloop/system_prompt.txt`) -- fix-ladder step 3, satisfied by
construction rather than by convention. `train_forgeloop_sft.py` already has
safe-interruption checkpointing, atomic saves, and a separate best.pt built
in (`docs/RESULTS.md` fix-ladder step 1: it also reports
`response_argmax_acc` and `first8_loss`, and flags `gate_ready` once val loss
<= 0.05 -- gate attempts below that are known to be a waste of a run).
7b polls and stops cleanly at `SFT_TRAIN_MINUTES` the same way 6c does for
base pretraining, so this stage is also "Run All"-safe and bounded.


In [ ]:
# 7) Launch ForgeLoop SFT as a background process, warm-started from the
#    base_35m checkpoint (or resumed from its own prior progress if SFT_CKPT
#    already exists).
SFT_TRAIN_MINUTES = 60 * 4  # not a trainer arg directly -- enforced by the wait() timeout below

sft_cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/train_forgeloop_sft.py"),
           "--base", str(BASE_CKPT), "--train", str(SCALED_TRAIN),
           "--val", str(SFT_DATA_DIR / "val.jsonl"), "--out", str(SFT_CKPT),
           "--system-file", str(SYSTEM_PROMPT_FILE),
           "--batch", "8", "--max-len", "1024", "--lr", "5e-5",
           "--eval-every", "25", "--checkpoint-every", "25",
           "--max-steps", "100000", "--patience", "40", "--balanced-domain-sampling"]

sft_log_fh = open(SFT_LOG, "a")
SFT_PROC = subprocess.Popen(sft_cmd, cwd=REPO_DIR, stdout=sft_log_fh, stderr=subprocess.STDOUT)
print("launched ForgeLoop SFT, PID", SFT_PROC.pid)
print("log:", SFT_LOG, "| checkpoint:", SFT_CKPT, "| best:", SFT_BEST)


In [ ]:
# 7a) Monitor SFT (rerun any time) -- parses the JSON records the trainer
#     already prints per eval, including the gate_ready flag.
alive = SFT_PROC.poll() is None
print("training alive:", alive, "" if alive else f"(exit code {SFT_PROC.returncode})")
records = []
for line in SFT_LOG.read_text(errors="replace").splitlines():
    line = line.strip()
    if line.startswith("{"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass
evals = [r for r in records if "val_loss" in r]
if evals:
    last = evals[-1]
    print(f"step {last['sft_step']} | val_loss {last['val_loss']:.4f} | "
          f"response_argmax_acc {last['response_argmax_acc']:.3f} | "
          f"first8_loss {last['first8_loss']:.4f} | "
          f"gate_ready {last['gate_ready']} | best_val {last['best_val_loss']:.4f}")
else:
    print("no eval rows yet")


In [ ]:
# 7b) Poll until SFT converges, hits max-steps, or SFT_TRAIN_MINUTES is spent
#     -- Run-All safe: prints progress and backs up to Drive automatically.
#     Sends SIGINT on timeout; the trainer's KeyboardInterrupt handler saves
#     before exiting, so this is a clean stop, not a kill.
import signal

POLL_SECONDS = 300
BACKUP_SECONDS = 1800
STOP_GRACE_SECONDS = 60

start = time.time()
last_backup = start
while True:
    try:
        rc = SFT_PROC.wait(timeout=POLL_SECONDS)
        print(f"SFT finished on its own, exit code {rc}")
        break
    except subprocess.TimeoutExpired:
        pass

    elapsed_min = (time.time() - start) / 60
    records = []
    for line in SFT_LOG.read_text(errors="replace").splitlines():
        line = line.strip()
        if line.startswith("{"):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    evals = [r for r in records if "val_loss" in r]
    if evals:
        last = evals[-1]
        print(f"[{elapsed_min:6.1f} min] step {last['sft_step']} | val_loss {last['val_loss']:.4f} | "
              f"response_argmax_acc {last['response_argmax_acc']:.3f} | "
              f"gate_ready {last['gate_ready']} | best_val {last['best_val_loss']:.4f}")
    else:
        print(f"[{elapsed_min:6.1f} min] no eval rows yet")

    if time.time() - last_backup >= BACKUP_SECONDS:
        for p in (SFT_CKPT, SFT_BEST):
            if p.exists():
                shutil.copy2(p, CKPT_ROOT / p.name)
        last_backup = time.time()
        print("    auto-backup done")

    if elapsed_min >= SFT_TRAIN_MINUTES:
        print(f"time budget ({SFT_TRAIN_MINUTES} min) reached -- stopping SFT cleanly")
        SFT_PROC.send_signal(signal.SIGINT)
        try:
            rc = SFT_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"stopped cleanly, exit code {rc}")
        except subprocess.TimeoutExpired:
            SFT_PROC.terminate()
            rc = SFT_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"forced stop after grace period, exit code {rc}")
        break

for p in (SFT_CKPT, SFT_BEST):
    if p.exists():
        shutil.copy2(p, CKPT_ROOT / p.name)

EVAL_CKPT = SFT_BEST if SFT_BEST.exists() else SFT_CKPT
ck = torch.load(EVAL_CKPT, map_location="cpu", weights_only=False)
print(f"\nevaluating from: {EVAL_CKPT}")
print(f"best_val_loss: {ck.get('best_val_loss'):.4f} (gate-ready threshold: 0.05)")


## 8) Score against the acceptance bar

Four checks that don't just re-measure aggregate loss, plus the sealed gate.
**Run the proof-gate cell exactly once per checkpoint and record the number
-- do not iterate against it.** If it fails, the next move is back to step 7
(more SFT / data), never editing the gate's 8 cases.


In [ ]:
# 8a) Choice-ranking MCQ accuracy vs. chance (measures knowledge even while
#     generation is still imperfect -- structure-before-content, per
#     docs/RESULTS.md).
from nueronce.chat import load_checkpoint
from nueronce.corpus.stack import get_entry
from nueronce.training.mcq_sft import load_and_convert, evaluate_mcq

model, _ = load_checkpoint(str(EVAL_CKPT))
model = model.to("cpu").eval()

MCQ_SUBJECTS = ["arc_easy", "arc_challenge", "openbookqa", "commonsense_qa", "math_qa", "gsm8k"]
mcq_results = {}
for subject in MCQ_SUBJECTS:
    records = list(load_and_convert(get_entry(subject), split="val", limit=200))
    if not records:
        records = list(load_and_convert(get_entry(subject), split="test", limit=200))
    if not records:
        print(f"{subject}: no held-out records available, skipping")
        continue
    result = evaluate_mcq(model, records, max_examples=100)
    mcq_results[subject] = result
    beats_chance = (result["accuracy"] - result["chance"]) * 100
    print(f"{subject:18s} acc {result['accuracy']:.3f}  chance {result['chance']:.3f}  "
          f"beats-chance {beats_chance:+.1f} pts  n={result['n']}")

n_beating_15 = sum(1 for r in mcq_results.values() if (r["accuracy"] - r["chance"]) * 100 >= 15)
print(f"\nsubjects beating chance by >=15 pts: {n_beating_15}/3 required -- "
      f"{'PASS' if n_beating_15 >= 3 else 'NOT YET'}")

Path("metrics").mkdir(exist_ok=True)
Path("metrics/colab_mcq_results.json").write_text(json.dumps(mcq_results, indent=2), encoding="utf-8")


In [ ]:
# 8b) Phase2 generative inference suite (>= 60% valid / non-echo /
#     stop-terminated is the acceptance bar).
subprocess.run([sys.executable, "-u", str(REPO_DIR / "scripts/eval_inference_phase2.py"),
                "--checkpoint", str(EVAL_CKPT),
                "--out", "metrics/colab_inference_phase2.json"],
               cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(Path("metrics/colab_inference_phase2.json").read_text(encoding="utf-8")),
                  indent=2)[:2000])


In [ ]:
# 8c) THE SEALED PROOF GATE. Run once per checkpoint, unmodified. Do not
#     rerun this repeatedly against the same checkpoint hoping for a
#     different number, and never edit the 8 cases in
#     scripts/eval_foundational_proof_gate.py to make this pass.
subprocess.run([sys.executable, "-u", str(REPO_DIR / "scripts/eval_foundational_proof_gate.py"),
                "--checkpoint", str(EVAL_CKPT),
                "--system-file", str(SYSTEM_PROMPT_FILE),
                "--output", "metrics/colab_foundational_proof_gate.json",
                "--no-fail-exit"],
               cwd=REPO_DIR, check=True)
gate = json.loads(Path("metrics/colab_foundational_proof_gate.json").read_text(encoding="utf-8"))
print(json.dumps(gate, indent=2)[:3000])


In [ ]:
# 8d) 5 raw transcripts, unedited -- the fluency bar (grammatical English
#     addressing the question), not the correctness bar.
from nueronce.chat import Conversation

conversation = Conversation(model, system=SYSTEM_PROMPT_FILE.read_text(encoding="utf-8"),
                             temperature=0.2, max_new=160, max_ctx=384)

for prompt in [
    "Hello. Introduce yourself in one sentence.",
    "What is 17 plus 26?",
    "Summarize the plan you would follow to fix a failing unit test.",
    "Rewrite this politely: 'send me the file now'.",
    "What should you do if you are not sure an answer is correct?",
]:
    print(">>>", prompt)
    print(conversation.say(prompt))
    print()


## 9) Acceptance scorecard + persist


In [ ]:
# 9) Consolidated scorecard from everything written to metrics/ above.
base_history = torch.load(BASE_CKPT, map_location="cpu", weights_only=False).get("history", [])
base_bpb = min((h["heldout_bpb"] for h in base_history), default=float("nan"))
sft_val_loss = ck.get("best_val_loss", float("nan"))
phase2 = json.loads(Path("metrics/colab_inference_phase2.json").read_text(encoding="utf-8"))
gate = json.loads(Path("metrics/colab_foundational_proof_gate.json").read_text(encoding="utf-8"))

print("=== ACCEPTANCE SCORECARD ===")
print(f"1) base_35m held-out bpb <= 1.5      : {base_bpb:.4f}  {'PASS' if base_bpb <= 1.5 else 'FAIL'}")
print(f"2) MCQ subjects beating chance >=15pt : {n_beating_15}/3  {'PASS' if n_beating_15 >= 3 else 'FAIL'}")
print(f"3) SFT response val_loss <= 0.05      : {sft_val_loss:.4f}  {'PASS' if sft_val_loss <= 0.05 else 'FAIL'}")
print("4) phase2 suite (see report above)    : inspect metrics/colab_inference_phase2.json for the pass rate")
print(f"5) sealed proof gate                  : {gate.get('overall', gate)}")
print()
print("A generalization dev set (>=100 unseen atomic examples, distinct from")
print("the training templates) is still owed per the advancement rules --")
print("data/foundational_val32 is the existing precedent but is sized for a")
print("probe, not the full requirement; build it out before declaring this done.")


In [ ]:
# 10) Persist everything to Drive. Git push is NOT automatic -- review the
#     diff yourself before pushing metrics/checkpoints from a Colab session.
for p in [BASE_CKPT, SFT_CKPT, SFT_BEST]:
    if p.exists():
        shutil.copy2(p, CKPT_ROOT / p.name)
for name in ["colab_mcq_results.json", "colab_inference_phase2.json", "colab_foundational_proof_gate.json"]:
    src = Path("metrics") / name
    if src.exists():
        shutil.copy2(src, DRIVE / "nueronce_checkpoints" / name)
print("backed up to", CKPT_ROOT)

print(f'''
To push results back (run manually, review the diff first):
  cd {REPO_DIR}
  git add metrics/colab_*.json
  git commit -m "colab: base_35m + forgeloop SFT run {VERSION}"
  git push origin {BRANCH}
''')
